In [ ]:
#!pip install prophet --quiet
#En mi caso uso colab

In [ ]:
# 1) CONFIGURACIÓN DE KAGGLE Y DESCARGA DEL DATASET
import os

kaggle_token_path = "/content/kaggle.json"
if os.path.exists(kaggle_token_path):
    !mkdir -p ~/.kaggle
    !cp /content/kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Se ha configurado correctamente el token de Kaggle.")
else:
    print("ADVERTENCIA: No se encontró kaggle.json, omite la descarga vía API.")

try:
    # Descarga del dataset desde Kaggle
    !kaggle datasets download -d marcosdiezz/dataset-3
    # Descomprimir
    !unzip -q *.zip
    print("Descarga y descompresión completa.")
except:
    print("No se pudo descargar automáticamente. Verifica tu token o descarga manualmente.")

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import matplotlib.pyplot as plt

# 3) LECTURA Y LIMPIEZA DE DATOS
file_path = "dataset_d3_filtrado.csv"

df = pd.read_csv(file_path)

print("Primeras filas del dataset original:")
print(df.head())

# Convertimos la columna 'datetime' a tipo fecha/hora
df['datetime'] = pd.to_datetime(df['datetime'])

# Eliminamos casas con 0 placas, importante con lo que habiamos hablado.
df = df[df['num_placas'] > 0]

# Ordenamos por fecha/hora por si no estuviera ordenado
df.sort_values(by='datetime', inplace=True)

# Agrupamos por hora para obtener la producción media de TODAS las casas
df_grouped = (
    df.groupby('datetime', as_index=False)
      .agg({'produccion_kWh': 'mean'})
      .rename(columns={'produccion_kWh': 'avg_produccion_kWh'})
)


In [ ]:
# 4) CÁLCULO DEL INCREMENTO/DECREMENTO PORCENTUAL POR HORA
df_grouped['avg_produccion_kWh_shift'] = df_grouped['avg_produccion_kWh'].shift(1)

# Evitamos divisiones por cero o NaNs iniciales
df_grouped.dropna(subset=['avg_produccion_kWh_shift'], inplace=True)

df_grouped['pct_change'] = (
    (df_grouped['avg_produccion_kWh'] - df_grouped['avg_produccion_kWh_shift'])
    / df_grouped['avg_produccion_kWh_shift']
) * 100

# ⚠️ Eliminar infinitos y NaNs resultantes
df_grouped = df_grouped.replace([np.inf, -np.inf], np.nan)
df_grouped.dropna(subset=['pct_change'], inplace=True)

# Nos quedamos solo con datetime y el % de cambio
df_for_prophet = df_grouped[['datetime', 'pct_change']].copy()
df_for_prophet.rename(columns={'datetime': 'ds', 'pct_change': 'y'}, inplace=True)

print("\nDatos finales que usaremos para Prophet (primeras filas):")
print(df_for_prophet.head())


In [ ]:
# 5) ENTRENAMIENTO DEL MODELO PROPHET
modelo = Prophet(
    daily_seasonality=True,   # por la fuerte estacionalidad a lo largo del día
    weekly_seasonality=False, # No sospechamos de variación semanal
    yearly_seasonality=True, # Abarcamos años asi que en true
    seasonality_mode='additive',
    changepoint_prior_scale=0.1,  # un poquito más alto que el default 0.05
    seasonality_prior_scale=10,   # default, ajustarlo si notamos sobreajuste
    interval_width=0.90           # un intervalo de confianza más amplio
)

modelo.fit(df_for_prophet)

In [ ]:
# 6) PREDICCIÓN (por ejemplo, para las próximas 24 horas)
# Creamos un dataframe futuro con las próximas 24 horas
# Prophet expandirá la serie hora a hora automáticamente.
future = modelo.make_future_dataframe(periods=24, freq='H')

# Realizamos la predicción
forecast = modelo.predict(future)

In [ ]:
# 7) VISUALIZACIÓN DE RESULTADOS
# La predicción estará en la columna 'yhat'
print("\nPredicciones generadas (últimas filas):")
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

# Plot de la predicción
fig1 = modelo.plot(forecast)
plt.title("Predicción de incremento/decremento porcentual de producción por hora")
plt.xlabel("Fecha/Hora")
plt.ylabel("Porcentaje de cambio (%)")
plt.show()

#Mostrar los componentes de la predicción
fig2 = modelo.plot_components(forecast)
plt.show()